In [ ]:
# ==============================================================================
# CureSense — MedGemma Service (Google Colab)
# colab_medgemma.ipynb
#
# Runs: MedGemma 1.5-4B for CT/MRI and X-ray image analysis
# GPU:  T4 (16 GB) — dedicated entirely to MedGemma
# URL:  MEDGEMMA_SERVICE_URL in Backend/.env
#
# Colab secrets required (left panel → key icon → Add new secret):
#   HF_TOKEN          — HuggingFace token (accept MedGemma terms first at
#                       hf.co/google/medgemma-1.5-4b-it)
#   NGROK_AUTH_TOKEN  — ngrok auth token (ngrok.com → Your Authtoken)
#
# Run all cells top-to-bottom. Copy the printed MEDGEMMA_SERVICE_URL into
# Backend/.env and restart Express.
# ==============================================================================

In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
import sys, os, subprocess

REPO_URL = 'https://github.com/moizaimran/curesense-project.git'
BRANCH   = 'hassan-branch'
REPO_DIR = '/content/curesense-project'

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)

In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
# Colab ships PyTorch (CUDA 13.x) and TorchAudio (CUDA 12.x) — version mismatch.
# MedGemma only does image analysis so TorchAudio is not needed at all.
# Remove it to prevent the CUDA version conflict from breaking AutoProcessor.
import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'torchaudio'], capture_output=True)
print('torchaudio removed (not needed for image analysis)')

# transformers >= 4.40 — AutoModelForImageTextToText + attn_implementation=sdpa
!pip install -q "transformers>=4.40.0"

# accelerate — required for device_map=
# pyngrok    — ngrok tunnel
# fastapi / uvicorn — MedGemma HTTP service
# pydicom    — DICOM ZIP extraction
# httpx      — async HTTP client
!pip install -q accelerate pyngrok fastapi uvicorn pydicom httpx

# pylibjpeg — JPEG Lossless compressed DICOMs (common in hospital exports)
!pip install -q pylibjpeg pylibjpeg-libjpeg

print('Installs complete')

In [ ]:
# ── Cell 3: Colab secrets + HuggingFace login + ngrok auth ───────────────────
#
# Add secrets in Colab: left panel → key icon → Add new secret
#   HF_TOKEN          — HuggingFace read token
#   NGROK_AUTH_TOKEN  — ngrok auth token
#
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok

login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

print('HuggingFace login OK')
print('ngrok ready')

In [ ]:
# ── Cell 4: Start MedGemma FastAPI on port 5002 ───────────────────────────────
#
# Colab T4 is the only GPU (GPU 0). _load_model() detects 1 GPU and
# automatically uses device_map='auto' — places everything on GPU 0.
#
import socket, threading, time, uvicorn
from api.medgemma_app import app as medgemma_app, _load_model

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

if _port_in_use(5002):
    print("[MedGemma] Already running on port 5002 — skipping restart")
else:
    threading.Thread(
        target=lambda: uvicorn.run(medgemma_app, host="0.0.0.0", port=5002, log_level="warning"),
        daemon=True,
    ).start()
    print("[MedGemma] FastAPI started on port 5002")

time.sleep(2)

# Pre-warm: downloads model weights (~8 GB) and prints VRAM stats
_load_model()

In [ ]:
# ── Cell 5: ngrok tunnel → MedGemma:5002 ─────────────────────────────────────
#
# Exposes the MedGemma FastAPI service at a public HTTPS URL.
# Copy the printed URL into Backend/.env as MEDGEMMA_SERVICE_URL.
#
from pyngrok import ngrok

# Disconnect stale tunnels from previous runs
for t in ngrok.get_tunnels():
    print(f"[ngrok] Disconnecting old tunnel: {t.public_url}")
    ngrok.disconnect(t.public_url)

PUBLIC_URL = ngrok.connect(5002).public_url
print(f"[ngrok] Connected → {PUBLIC_URL} → MedGemma:5002")

print("\n" + "=" * 64)
print(f"  MEDGEMMA_SERVICE_URL = {PUBLIC_URL}")
print("=" * 64)
print("\nPaste into Backend/.env as MEDGEMMA_SERVICE_URL and restart Express.")
print("The Kaggle notebook provides AI_SERVICE_URL (Flask — PDF/interview).")